# London Strategic Edge Market Data Notebook

Batch-fetch candles from London Strategic Edge for crypto, Nasdaq stocks, and B3 stocks. The notebook is safe to run without credentials: it only performs API requests when `LSE_API_KEY` is available in the environment.

## API Notes From Official Sources

- Python client package: `pip install lse-data`, imported as `from lse import LSE`.
- API key: pass `LSE(api_key="...")` or set `LSE_API_KEY` in the environment.
- Historical candles: `client.candles(symbol, timeframe, start=..., end=..., limit=..., order=...)`.
- Bulk history: `client.history(symbol, timeframe=..., start=...)`.
- Catalog discovery: `client.catalog()` or `client.catalog("stocks")`, `client.catalog("crypto")`, etc.
- Timeframes documented by the client README: `1s`, `5s`, `15s`, `30s`, `1m`, `3m`, `5m`, `15m`, `30m`, `1h`, `4h`, `1d`, `1w`, `1mo`.

References:

- https://github.com/londonstrategicedge/lse-data
- https://londonstrategicedge.com/api-documentation/
- https://londonstrategicedge.com/data/

In [1]:
from __future__ import annotations

import os
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd

try:
    from lse import LSE, LSEError
    LSE_AVAILABLE = True
except ModuleNotFoundError:
    LSE = None
    LSEError = Exception
    LSE_AVAILABLE = False

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print(f"lse-data installed: {LSE_AVAILABLE}")
if not LSE_AVAILABLE:
    print("Install it with: pip install lse-data")

lse-data installed: True


In [2]:
@dataclass(frozen=True)
class FetchConfig:
    timeframe: str = "1d"
    start: str = "2025-01-01"
    end: str | None = None
    limit: int | None = None
    order: str = "asc"
    sleep_seconds: float = 0.20
    output_dir: Path = Path("data/lse_market_data")
    write_parquet: bool = True
    write_csv: bool = True
    use_catalog_resolution: bool = True


config = FetchConfig()
config.output_dir.mkdir(parents=True, exist_ok=True)

def load_dotenv_if_needed(path: Path = Path(".env")) -> None:
    if os.getenv("LSE_API_KEY") or not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


load_dotenv_if_needed()
API_KEY = os.getenv("LSE_API_KEY")
RUN_LIVE_REQUESTS = bool(API_KEY and LSE_AVAILABLE)

print(config)
print(f"LSE_API_KEY present: {bool(API_KEY)}")
print(f"RUN_LIVE_REQUESTS: {RUN_LIVE_REQUESTS}")

FetchConfig(timeframe='1d', start='2025-01-01', end=None, limit=None, order='asc', sleep_seconds=0.2, output_dir=PosixPath('data/lse_market_data'), write_parquet=True, write_csv=True, use_catalog_resolution=True)
LSE_API_KEY present: True
RUN_LIVE_REQUESTS: True


In [3]:
CRYPTO_SYMBOLS = [
    "BTC/USD", "ETH/USD", "SOL/USD", "XRP/USD", "DOGE/USD",
    "BNB/USD", "ADA/USD", "AVAX/USD", "LINK/USD", "DOT/USD",
]

NASDAQ_SYMBOLS = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
    "META", "TSLA", "AVGO", "COST", "NFLX",
    "AMD", "INTC", "ADBE", "QCOM", "CSCO",
]

# Local B3 tickers often appear as PETR4.SA in public data vendors. In the current
# LSE catalog, local B3 symbols may be absent, so the resolver below falls back to
# Brazil ADR/US-listed Brazil exposure names that are present in the catalog.
B3_SYMBOLS = [
    "PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA",
    "WEGE3.SA", "BBAS3.SA", "B3SA3.SA", "PRIO3.SA", "RENT3.SA",
]

BRAZIL_ADR_FALLBACK_SYMBOLS = [
    "PBR", "PBR.A", "VALE", "ITUB", "BBD", "ABEV", "NU", "PAGS",
    "XP", "SBS", "SUZ", "GGB", "VIV", "UGP", "BSBR", "CSAN",
    "TIMB", "SID", "CIG",
]

REQUESTED_UNIVERSE = {
    "crypto": CRYPTO_SYMBOLS,
    "nasdaq": NASDAQ_SYMBOLS,
    "b3": B3_SYMBOLS,
}

for group, symbols in REQUESTED_UNIVERSE.items():
    print(f"{group:>7}: {len(symbols)} symbols")

 crypto: 10 symbols
 nasdaq: 15 symbols
     b3: 10 symbols


In [4]:
def make_client() -> Any | None:
    if not LSE_AVAILABLE:
        print("lse-data is not installed. Live calls disabled.")
        return None
    if not API_KEY:
        print("LSE_API_KEY is not set. Live calls disabled.")
        return None
    return LSE(api_key=API_KEY)


client = make_client()
print(f"Client ready: {client is not None}")

Client ready: True


In [5]:
def safe_filename(symbol: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", symbol).strip("-")


def rows_to_frame(rows: list[dict[str, Any]], symbol: str, group: str) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df.insert(0, "asset_group", group)
    if "symbol" not in df.columns:
        df.insert(1, "symbol", symbol)
    for col in ["timestamp", "time", "date", "datetime"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
    return df


def write_frame(df: pd.DataFrame, symbol: str, group: str, cfg: FetchConfig) -> dict[str, Path]:
    group_dir = cfg.output_dir / group
    group_dir.mkdir(parents=True, exist_ok=True)
    stem = safe_filename(symbol)
    paths: dict[str, Path] = {}
    if cfg.write_parquet:
        path = group_dir / f"{stem}_{cfg.timeframe}.parquet"
        df.to_parquet(path, index=False)
        paths["parquet"] = path
    if cfg.write_csv:
        path = group_dir / f"{stem}_{cfg.timeframe}.csv"
        df.to_csv(path, index=False)
        paths["csv"] = path
    return paths


def load_catalog(client: Any | None, category: str) -> pd.DataFrame:
    if client is None:
        return pd.DataFrame()
    try:
        return pd.DataFrame(client.catalog(category))
    except LSEError as exc:
        print(f"catalog({category}) failed: {getattr(exc, 'status', '')} {getattr(exc, 'message', exc)}")
        return pd.DataFrame()


def candidate_symbols(symbol: str) -> list[str]:
    base = symbol.strip().upper()
    candidates = [base]
    if base.endswith(".SA"):
        candidates.append(base[:-3])
    else:
        candidates.append(f"{base}.SA")
    candidates.append(base.replace("/", "-"))
    candidates.append(base.replace("-", "/"))
    return list(dict.fromkeys(candidates))


def resolve_symbols(requested: list[str], catalog_df: pd.DataFrame) -> tuple[list[str], list[str]]:
    if catalog_df.empty or "symbol" not in catalog_df.columns:
        return requested, []
    available = {str(s).upper(): str(s) for s in catalog_df["symbol"].dropna()}
    resolved: list[str] = []
    missing: list[str] = []
    for requested_symbol in requested:
        match = None
        for candidate in candidate_symbols(requested_symbol):
            if candidate.upper() in available:
                match = available[candidate.upper()]
                break
        if match:
            resolved.append(match)
        else:
            missing.append(requested_symbol)
    return resolved, missing


def fetch_candles_one(client: Any, symbol: str, group: str, cfg: FetchConfig) -> pd.DataFrame:
    kwargs = {"start": cfg.start, "order": cfg.order}
    if cfg.end is not None:
        kwargs["end"] = cfg.end
    if cfg.limit is not None:
        kwargs["limit"] = cfg.limit
    rows = client.candles(symbol, cfg.timeframe, **kwargs)
    return rows_to_frame(rows, symbol=symbol, group=group)


def fetch_candles_many(client: Any | None, universe: dict[str, list[str]], cfg: FetchConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    if client is None:
        print("Dry run only. Set LSE_API_KEY and install lse-data to fetch.")
        plan = [{"asset_group": group, "symbol": symbol, "timeframe": cfg.timeframe, "start": cfg.start, "end": cfg.end} for group, symbols in universe.items() for symbol in symbols]
        return pd.DataFrame(), pd.DataFrame(plan)

    frames: list[pd.DataFrame] = []
    failures: list[dict[str, Any]] = []
    for group, symbols in universe.items():
        for symbol in symbols:
            try:
                df = fetch_candles_one(client, symbol, group, cfg)
                if not df.empty:
                    write_frame(df, symbol, group, cfg)
                    frames.append(df)
                print(f"OK {group:>7} {symbol:<12} rows={len(df):,}")
            except LSEError as exc:
                failures.append({"asset_group": group, "symbol": symbol, "status": getattr(exc, "status", None), "message": getattr(exc, "message", str(exc))})
                print(f"FAIL {group:>7} {symbol:<12} {getattr(exc, 'message', exc)}")
            except Exception as exc:
                failures.append({"asset_group": group, "symbol": symbol, "status": None, "message": str(exc)})
                print(f"FAIL {group:>7} {symbol:<12} {exc}")
            time.sleep(cfg.sleep_seconds)

    data = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return data, pd.DataFrame(failures)

In [6]:
resolved_universe = REQUESTED_UNIVERSE.copy()
missing_by_group: dict[str, list[str]] = {}

if client is not None and config.use_catalog_resolution:
    crypto_catalog = load_catalog(client, "crypto")
    stock_catalog = load_catalog(client, "stocks")

    resolved_universe["crypto"], missing_by_group["crypto"] = resolve_symbols(CRYPTO_SYMBOLS, crypto_catalog)
    resolved_universe["nasdaq"], missing_by_group["nasdaq"] = resolve_symbols(NASDAQ_SYMBOLS, stock_catalog)
    resolved_universe["b3"], missing_by_group["b3"] = resolve_symbols(B3_SYMBOLS, stock_catalog)
    if not resolved_universe["b3"]:
        print("No local B3 symbols were found in the LSE stocks catalog; using Brazil ADR/catalog fallback symbols.")
        resolved_universe["b3"], fallback_missing = resolve_symbols(BRAZIL_ADR_FALLBACK_SYMBOLS, stock_catalog)
        missing_by_group["b3_fallback"] = fallback_missing
else:
    print("Catalog resolution skipped. Running with requested symbols.")

for group, symbols in resolved_universe.items():
    print(f"{group:>7}: {len(symbols)} resolved symbols")
    if missing_by_group.get(group):
        print(f"        local/requested missing: {missing_by_group[group]}")
    if group == "b3" and missing_by_group.get("b3_fallback"):
        print(f"        fallback missing: {missing_by_group['b3_fallback']}")

No local B3 symbols were found in the LSE stocks catalog; using Brazil ADR/catalog fallback symbols.
 crypto: 10 resolved symbols
 nasdaq: 15 resolved symbols
     b3: 19 resolved symbols
        local/requested missing: ['PETR4.SA', 'VALE3.SA', 'ITUB4.SA', 'BBDC4.SA', 'ABEV3.SA', 'WEGE3.SA', 'BBAS3.SA', 'B3SA3.SA', 'PRIO3.SA', 'RENT3.SA']


## Minimal Snippet

Use this outside the notebook after setting `LSE_API_KEY`.

In [7]:
snippet = r'''
import os
import pandas as pd
from lse import LSE

client = LSE(api_key=os.environ["LSE_API_KEY"])

symbols = [
    "BTC/USD", "ETH/USD", "SOL/USD",
    "AAPL", "MSFT", "NVDA", "AMZN",
    "PETR4.SA", "VALE3.SA", "ITUB4.SA",
]

frames = []
for symbol in symbols:
    rows = client.candles(symbol, "1d", start="2025-01-01", order="asc")
    df = pd.DataFrame(rows)
    df["symbol"] = symbol
    frames.append(df)

market_data = pd.concat(frames, ignore_index=True)
market_data.to_parquet("lse_market_data.parquet", index=False)
'''

print(snippet)


import os
import pandas as pd
from lse import LSE

client = LSE(api_key=os.environ["LSE_API_KEY"])

symbols = [
    "BTC/USD", "ETH/USD", "SOL/USD",
    "AAPL", "MSFT", "NVDA", "AMZN",
    "PETR4.SA", "VALE3.SA", "ITUB4.SA",
]

frames = []
for symbol in symbols:
    rows = client.candles(symbol, "1d", start="2025-01-01", order="asc")
    df = pd.DataFrame(rows)
    df["symbol"] = symbol
    frames.append(df)

market_data = pd.concat(frames, ignore_index=True)
market_data.to_parquet("lse_market_data.parquet", index=False)



In [8]:
market_data, failures_or_plan = fetch_candles_many(client, resolved_universe, config)

if market_data.empty:
    print("No live market data fetched in this run.")
    display(failures_or_plan.head(20))
else:
    combined_path = config.output_dir / f"combined_{config.timeframe}.parquet"
    market_data.to_parquet(combined_path, index=False)
    print(f"Combined rows: {len(market_data):,}")
    print(f"Saved combined parquet: {combined_path}")
    display(market_data.head())
    if not failures_or_plan.empty:
        display(failures_or_plan)

OK  crypto BTC/USD      rows=557


OK  crypto ETH/USD      rows=526


OK  crypto SOL/USD      rows=557


OK  crypto XRP/USD      rows=495


OK  crypto DOGE/USD     rows=557


...

OK      b3 TIMB         rows=50


OK      b3 SID          rows=46


OK      b3 CIG          rows=46


Combined rows: 12,577
Saved combined parquet: data/lse_market_data/combined_1d.parquet


,asset_group,symbol,open,high,low,close,volume,timestamp
0,crypto,BTC/USD,93575.99,95151.15,92887.99,94591.79,10373.32613,2025-01-01 00:00:00+00:00
1,crypto,BTC/USD,94591.78,97839.51,94391.99,96984.80,21970.48948,2025-01-02 00:00:00+00:00
2,crypto,BTC/USD,96984.79,98976.92,96100.00,98174.18,15253.82936,2025-01-03 00:00:00+00:00
3,crypto,BTC/USD,98174.17,98778.43,97514.78,98220.49,8990.05651,2025-01-04 00:00:00+00:00
4,crypto,BTC/USD,98220.51,98836.86,97276.78,98363.61,8095.63723,2025-01-05 00:00:00+00:00


In [9]:
if not market_data.empty:
    summary = (
        market_data.groupby(["asset_group", "symbol"], dropna=False)
        .size()
        .rename("rows")
        .reset_index()
        .sort_values(["asset_group", "symbol"])
    )
    display(summary)
else:
    print("Quick checks skipped because this was a dry run.")

,asset_group,symbol,rows
0,b3,ABEV,50
1,b3,BBD,47
2,b3,BSBR,50
3,b3,CIG,46
4,b3,CSAN,49
5,b3,GGB,49
6,b3,ITUB,49
7,b3,NU,404
8,b3,PAGS,405
9,b3,PBR,50
